In [16]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split

In [17]:
# loading and printing the first 5 rows of the dataset
data = pd.read_csv("melb_data.csv")
data.head(5)

,Suburb,Address,Rooms,Type,Price,Method,SellerG,Date,Distance,Postcode,...,Bathroom,Car,Landsize,BuildingArea,YearBuilt,CouncilArea,Lattitude,Longtitude,Regionname,Propertycount
0,Abbotsford,85 Turner St,2,h,1480000.0,S,Biggin,3/12/2016,2.5,3067.0,...,1.0,1.0,202.0,NaN,NaN,Yarra,-37.7996,144.9984,Northern Metropolitan,4019.0
1,Abbotsford,25 Bloomburg St,2,h,1035000.0,S,Biggin,4/02/2016,2.5,3067.0,...,1.0,0.0,156.0,79.0,1900.0,Yarra,-37.8079,144.9934,Northern Metropolitan,4019.0
2,Abbotsford,5 Charles St,3,h,1465000.0,SP,Biggin,4/03/2017,2.5,3067.0,...,2.0,0.0,134.0,150.0,1900.0,Yarra,-37.8093,144.9944,Northern Metropolitan,4019.0
3,Abbotsford,40 Federation La,3,h,850000.0,PI,Biggin,4/03/2017,2.5,3067.0,...,2.0,1.0,94.0,NaN,NaN,Yarra,-37.7969,144.9969,Northern Metropolitan,4019.0
4,Abbotsford,55a Park St,4,h,1600000.0,VB,Nelson,4/06/2016,2.5,3067.0,...,1.0,2.0,120.0,142.0,2014.0,Yarra,-37.8072,144.9941,Northern Metropolitan,4019.0


In [18]:
# checking the amount of rows and columns
data.shape

(13580, 21)

In [19]:
#inspecting and fixing Wrong column data types https://pbpython.com/pandas_dtypes.html
print(data.info())
data["Date"] = pd.to_datetime(data["Date"],errors = "coerce")
data["BuildingArea"] = pd.to_numeric(data["BuildingArea"],errors = "coerce")
data["YearBuilt"] = pd.to_numeric(data["YearBuilt"],errors = "coerce")
data["Propertycount"] = pd.to_numeric(data["Propertycount"],errors = "coerce")

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 13580 entries, 0 to 13579
Data columns (total 21 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Suburb         13580 non-null  object 
 1   Address        13580 non-null  object 
 2   Rooms          13580 non-null  int64  
 3   Type           13580 non-null  object 
 4   Price          13580 non-null  float64
 5   Method         13580 non-null  object 
 6   SellerG        13580 non-null  object 
 7   Date           13580 non-null  object 
 8   Distance       13580 non-null  float64
 9   Postcode       13580 non-null  float64
 10  Bedroom2       13580 non-null  float64
 11  Bathroom       13580 non-null  float64
 12  Car            13518 non-null  float64
 13  Landsize       13580 non-null  float64
 14  BuildingArea   7130 non-null   float64
 15  YearBuilt      8205 non-null   float64
 16  CouncilArea    12211 non-null  object 
 17  Lattitude      13580 non-null  float64
 18  Longti

In [20]:
# summing up all the null values in the dataset
data.isnull().sum()

,0
Suburb,0
Address,0
Rooms,0
Type,0
Price,0
Method,0
SellerG,0
Date,8442
Distance,0
Postcode,0


In [21]:
# checking if the mean > median are  to decide whether to use mean or median imputation
data[["Bedroom2", "Bathroom", "Car", "Landsize", "Lattitude", "Longtitude"]].describe()

,Bedroom2,Bathroom,Car,Landsize,Lattitude,Longtitude
count,13580.000000,13580.000000,13518.000000,13580.000000,13580.000000,13580.000000
mean,2.914728,1.534242,1.610075,558.416127,-37.809203,144.995216
std,0.965921,0.691712,0.962634,3990.669241,0.079260,0.103916
min,0.000000,0.000000,0.000000,0.000000,-38.182550,144.431810
25%,2.000000,1.000000,1.000000,177.000000,-37.856822,144.929600
50%,3.000000,1.000000,2.000000,440.000000,-37.802355,145.000100
75%,3.000000,2.000000,2.000000,651.000000,-37.756400,145.058305
max,20.000000,8.000000,10.000000,433014.000000,-37.408530,145.526350


In [22]:
#drop columns that are mostly missing and not business-critical page 52
data = data.drop(columns=["BuildingArea", "YearBuilt", "Date"])

#drop rows where missingness is rare page 52
data = data.dropna(subset=["Distance", "Postcode", "CouncilArea", "Propertycount"])

#drop rows where critical label Price is missing and cannot be reliably imputed page 52
data = data.dropna(subset=["Price"])

In [23]:
# according to slide 192, a right-skewed distribution is identified when the mean is greater than the median.
# as stated on Page 53, median imputation is best for skewed distributions as it is robust to outliers and preserves the center better.
# Therefore I used median imputation for Landsize.
#I also found this rule while seraching online,
 #mean=median use mean imputaion because they are symmetric,
# mean >/< median use median imputation, because they are asymmetric
data["Landsize"] = data["Landsize"].fillna(data["Landsize"].median())
# a mean of 593 and a median of 521 so we use median imputation
data["Bedroom2"]=data["Bedroom2"].fillna(data["Bedroom2"].mean())
# a mean of 3.08 and a median of 3.0 so we use mean because the are symmetric
data["Bathroom"]=data["Bathroom"].fillna(data["Bathroom"].median())
# a mean of 1.62 and a median of 2.0, so we use median imputaion
data["Car"]=data["Car"].fillna(data["Car"].median())
# a mean of 1.73 and a median of 2.0 so we use median imputation
data["Lattitude"] = data["Lattitude"].fillna(data["Lattitude"].mean())
# a mean and median of -37.81 so we use mean
data["Longtitude"] = data["Longtitude"].fillna(data["Longtitude"].mean())
# a mean and median of 145.0 so we use mean imputaion

# each comment belongs to the line of code above it

In [24]:
data.duplicated().sum()
#checking if there are duplicates to drop

np.int64(16)

In [25]:
data = data.drop_duplicates()


In [26]:
data.duplicated().sum()
# it worked

np.int64(0)

In [27]:
# checking for outliers using the iqr method
Q1 = data["Price"].quantile(.25)
Q3 = data["Price"].quantile(.75)

IQR = Q3 - Q1
lower = Q1 - 1.5 * IQR
upper = Q3 + 1.5 * IQR
print(lower)
print(upper)

#we use the iqr method to spot outliers in the price column,
# according to slide 70 we use iqr because it is based on quartiles making it less affected and
# more reliable than mean and standard deviation for skewed data such as house prices, day 10, page 70

upper_99 = data["Price"].quantile(.99)
data["Price"] = data["Price"].clip(upper = upper_99)
#according to slide 83 we cap the values at the 99th percentile using .clip
# limiting the influence of extreme house prices without removing the values entirely
# slides 79 states capping is appropriate when tails are real but should not dominate the model.

-380000.0
2340000.0


In [28]:
data = pd.read_csv("melb_data.csv")
def clean_data(data):
    data = data.copy()

    data["Date"] = pd.to_datetime(data["Date"], errors="coerce")
    data["BuildingArea"] = pd.to_numeric(data["BuildingArea"], errors="coerce")
    data["YearBuilt"] = pd.to_numeric(data["YearBuilt"], errors="coerce")
    data["Propertycount"] = pd.to_numeric(data["Propertycount"], errors="coerce")

    data = data.drop(columns=["BuildingArea", "YearBuilt", "Date"])

    data = data.dropna(subset=["Distance", "Postcode", "CouncilArea", "Propertycount"])

    data = data.dropna(subset=["Price"])

    data["Landsize"] = data["Landsize"].fillna(data["Landsize"].median())
    data["Bedroom2"] = data["Bedroom2"].fillna(data["Bedroom2"].median())
    data["Bathroom"] = data["Bathroom"].fillna(data["Bathroom"].mean())
    data["Car"] = data["Car"].fillna(data["Car"].mean())
    data["Lattitude"] = data["Lattitude"].fillna(data["Lattitude"].mean())
    data["Longtitude"] = data["Longtitude"].fillna(data["Longtitude"].mean())

    data = data.drop_duplicates()

    Q1 = data["Price"].quantile(0.25)
    Q3 = data["Price"].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    print(lower)
    print(upper)
    data = data[(data["Price"] >= lower) & (data["Price"] <= upper)]

    return data

data = clean_data(data)

# the clean_data() function  based on slide 97  states that putting all cleaning
# steps into a single reusable function makes the code reproducible and easier to debug.
# slide 99 specifies the order of operations, types first, then missing values, then duplicates, then outliers.

-380000.0
2340000.0


In [29]:
assert data["Price"].isna().sum() == 0
assert data["Price"].min() > 0
assert data.shape[1] == 18
#page 100 of the slides says we should check key invariants after cleaning to make sure everything worked.
#the first check makes sure Price has no missing values left.
#the second check makes sure no house has a price of 0 or below, which would be a data error.
#the third check makes sure the dataset has the right number of columns after dropping some earlier.

In [30]:
# save cleaned data
data.to_csv("melbo_clean.csv", index=False)